# 01 — Dataset Preprocessing & EDA
---
This notebook handles:
1. **Unzipping** raw CWRU and Paderborn bearing datasets
2. **Sliding-window extraction** with Z-score normalization
3. **Saving** processed `.npz` files into `../datasets/processed/`

In [ ]:
import os
import zipfile
import numpy as np
import scipy.io
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

DATASETS_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "datasets"))
PROCESSED_DIR = os.path.join(DATASETS_ROOT, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Datasets root : {DATASETS_ROOT}")
print(f"Processed dir : {PROCESSED_DIR}")

## 1 — Unzip Raw Datasets

In [ ]:
# --- Unzip CWRU ---
cwru_zip = os.path.join(DATASETS_ROOT, "cwru-bearing-datasets.zip")
cwru_extract_dir = os.path.join(DATASETS_ROOT, "cwru_raw")
if not os.path.exists(cwru_extract_dir):
    print(f"Extracting {cwru_zip} ...")
    with zipfile.ZipFile(cwru_zip, "r") as zf:
        zf.extractall(cwru_extract_dir)
    print("Done.")
else:
    print(f"CWRU already extracted to {cwru_extract_dir}")

# --- Unzip Paderborn ---
pb_zip = os.path.join(DATASETS_ROOT, "paderborn_bearing_dataset.zip")
pb_extract_dir = os.path.join(DATASETS_ROOT, "paderborn_raw")
if not os.path.exists(pb_extract_dir):
    print(f"Extracting {pb_zip} ...")
    with zipfile.ZipFile(pb_zip, "r") as zf:
        zf.extractall(pb_extract_dir)
    print("Done.")
else:
    print(f"Paderborn already extracted to {pb_extract_dir}")

print("\nCWRU contents:", os.listdir(cwru_extract_dir))
print("Paderborn contents:", os.listdir(pb_extract_dir))

## 2 — Process CWRU Bearing Dataset
- **Signal:** Drive-End (DE) accelerometer
- **Window:** 2048 samples, Step: 512
- **Normalization:** Z-score per feature
- **Labels:** 10 classes (1 Normal + 9 Fault severities × locations)

In [ ]:
WINDOW = 2048
STEP   = 512

# CWRU .mat filename → (label_index, readable_name)
# Label 0 = Normal, Labels 1-9 = Fault classes
CWRU_LABEL_MAP = {
    "Time_Normal_1_098":  (0, "Normal"),
    "B007_1_123":         (1, "Ball-0.007"),
    "B014_1_190":         (2, "Ball-0.014"),
    "B021_1_227":         (3, "Ball-0.021"),
    "IR007_1_110":        (4, "IR-0.007"),
    "IR014_1_175":        (5, "IR-0.014"),
    "IR021_1_214":        (6, "IR-0.021"),
    "OR007_6_1_136":      (7, "OR-0.007"),
    "OR014_6_1_202":      (8, "OR-0.014"),
    "OR021_6_1_239":      (9, "OR-0.021"),
}

raw_dir = os.path.join(cwru_extract_dir, "raw")

all_windows = []
all_labels  = []

for fname, (label, name) in CWRU_LABEL_MAP.items():
    mat_path = os.path.join(raw_dir, f"{fname}.mat")
    if not os.path.exists(mat_path):
        print(f"  [SKIP] {mat_path} not found"); continue

    mat = scipy.io.loadmat(mat_path)
    # Pick the Drive-End (DE) accelerometer key
    de_key = [k for k in mat.keys() if "DE_time" in k]
    if not de_key:
        de_key = [k for k in mat.keys() if not k.startswith("__")]
        de_key = [de_key[0]] if de_key else []
    if not de_key:
        print(f"  [SKIP] No suitable key in {fname}"); continue

    signal = mat[de_key[0]].flatten()
    n_windows = (len(signal) - WINDOW) // STEP + 1
    for i in range(n_windows):
        seg = signal[i * STEP : i * STEP + WINDOW]
        all_windows.append(seg)
        all_labels.append(label)

    print(f"  {name:12s}  signal={len(signal):>8,}  windows={n_windows}")

X = np.array(all_windows, dtype=np.float32)  # (N, 2048)
y = np.array(all_labels, dtype=np.int64)

# Z-score normalization (per-sample, channel=1)
scaler = StandardScaler()
X = scaler.fit_transform(X)          # (N, 2048) — already flat for 1-ch
X = X[:, :, np.newaxis]              # (N, 2048, 1)

print(f"\nCWRU Processed — X: {X.shape}, y: {y.shape}, classes: {np.unique(y)}")

# Stratified split 80/10/10
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

save_path = os.path.join(PROCESSED_DIR, "cwru_processed.npz")
np.savez_compressed(save_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test)
print(f"Saved → {save_path}  (train={len(y_train)}, val={len(y_val)}, test={len(y_test)})")

## 3 — Process Paderborn Bearing Dataset
- **Window:** 4096 samples
- **Normalization:** Z-score
- **Labels:** 32 classes (6 healthy + 12 artificial damage + 14 real damage)
- **Requires:** `patool` or `unrar` to extract inner `.rar` archives

In [ ]:
import subprocess, glob

PB_WINDOW = 4096
PB_STEP   = 1024
pb_mat_dir = os.path.join(pb_extract_dir, "mat_files")
os.makedirs(pb_mat_dir, exist_ok=True)

# Step 1: Extract every .rar → individual .mat files
rar_files = sorted(glob.glob(os.path.join(pb_extract_dir, "*.rar")))
print(f"Found {len(rar_files)} .rar archives")

for rar_path in rar_files:
    bearing_name = os.path.splitext(os.path.basename(rar_path))[0]
    out_dir = os.path.join(pb_mat_dir, bearing_name)
    if os.path.exists(out_dir) and os.listdir(out_dir):
        continue
    os.makedirs(out_dir, exist_ok=True)
    try:
        import patoolib
        patoolib.extract_archive(rar_path, outdir=out_dir, verbosity=-1)
    except ImportError:
        subprocess.run(["unrar", "x", "-o+", rar_path, out_dir],
                       capture_output=True, check=True)
    print(f"  Extracted {bearing_name}")

# Step 2: Assign labels (0-31) alphabetically by bearing folder
bearing_dirs = sorted([d for d in os.listdir(pb_mat_dir)
                       if os.path.isdir(os.path.join(pb_mat_dir, d))])
label_map = {name: idx for idx, name in enumerate(bearing_dirs)}
print(f"\nPaderborn bearings: {len(bearing_dirs)} → labels 0-{len(bearing_dirs)-1}")

# Step 3: Sliding-window extraction
all_windows_pb, all_labels_pb = [], []

for bearing_name, label in label_map.items():
    mat_files = glob.glob(os.path.join(pb_mat_dir, bearing_name, "**", "*.mat"), recursive=True)
    total_wins = 0
    for mf in mat_files:
        mat = scipy.io.loadmat(mf)
        # Paderborn .mat files typically store vibration under a structured key
        for key in mat:
            if key.startswith("__"):
                continue
            data = mat[key]
            # Navigate nested structured arrays
            if hasattr(data, "dtype") and data.dtype.names:
                for sub in data.dtype.names:
                    arr = data[sub].flatten()
                    if hasattr(arr[0], "dtype") and arr[0].dtype.names:
                        for sub2 in arr[0].dtype.names:
                            inner = arr[0][sub2].flatten()
                            if inner.dtype.kind == "f" and len(inner) > PB_WINDOW:
                                sig = inner.astype(np.float32)
                                n_w = (len(sig) - PB_WINDOW) // PB_STEP + 1
                                for i in range(n_w):
                                    all_windows_pb.append(sig[i*PB_STEP:i*PB_STEP+PB_WINDOW])
                                    all_labels_pb.append(label)
                                total_wins += n_w
            elif data.dtype.kind == "f" and data.size > PB_WINDOW:
                sig = data.flatten().astype(np.float32)
                n_w = (len(sig) - PB_WINDOW) // PB_STEP + 1
                for i in range(n_w):
                    all_windows_pb.append(sig[i*PB_STEP:i*PB_STEP+PB_WINDOW])
                    all_labels_pb.append(label)
                total_wins += n_w

    if total_wins:
        print(f"  {bearing_name:6s} → label {label:2d}, windows={total_wins}")

if all_windows_pb:
    X_pb = np.array(all_windows_pb, dtype=np.float32)
    y_pb = np.array(all_labels_pb, dtype=np.int64)

    scaler_pb = StandardScaler()
    X_pb = scaler_pb.fit_transform(X_pb)
    X_pb = X_pb[:, :, np.newaxis]  # (N, 4096, 1)

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_pb, y_pb, test_size=0.2, stratify=y_pb, random_state=42)
    X_v, X_te, y_v, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

    save_path_pb = os.path.join(PROCESSED_DIR, "paderborn_processed.npz")
    np.savez_compressed(save_path_pb,
        X_train=X_tr, y_train=y_tr,
        X_val=X_v,    y_val=y_v,
        X_test=X_te,  y_test=y_te)
    print(f"\nSaved → {save_path_pb}  (train={len(y_tr)}, val={len(y_v)}, test={len(y_te)})")
else:
    print("\n⚠️  No Paderborn windows extracted — check .rar extraction (install patool or unrar)")